# 07 PII, Masking and Row Filters

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Protect sensitive data at the row and column level: <b>dynamic views</b> that mask columns and filter rows by user or group, Unity Catalog <b>column masks</b> and <b>row filters</b> attached directly to tables, mapping tables for row-level access, and pseudonymization with hashing. Know which approach to choose.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Regulations like GDPR and HIPAA, and simple common sense, require that people see only the personal data they need. Unity Catalog lets you enforce that <b>in the platform</b>, consistently for every tool, instead of trusting each notebook and dashboard to hide columns. Row filters and column masks are explicitly listed in the exam's governance domain.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Options for fine-grained access</b><br>
| Approach | How | Pros | Cons |<br>|---|---|---|---|<br>| Separate view | View without sensitive columns | Simple | One view per audience; base table must stay locked |<br>| Dynamic view | View with <code>CASE WHEN is_account_group_member(...)</code> and <code>WHERE</code> on <code>current_user()</code> | Flexible, one view for everyone | Users must query the view, not the table |<br>| <b>Column mask</b> | SQL function attached to a column: <code>ALTER COLUMN c SET MASK f</code> | Enforced on the table itself, for every query and tool | One function per masking rule |<br>| <b>Row filter</b> | Boolean SQL function attached to a table: <code>SET ROW FILTER f ON (col)</code> | Enforced on the table itself | Adds query overhead; keep functions simple |<br>| Pseudonymization | Replace values with hashes (<code>sha2</code>) at write time | Joins still work, irreversible | Data changed permanently; not access control |<br><br>
Useful functions inside masks, filters and dynamic views:
<ul><li><code>current_user()</code>: the email (or ID) of the user running the query</li><li><code>is_account_group_member('group')</code>: is the user in an account-level group?</li><li>Mapping tables: <code>EXISTS (SELECT 1 FROM access_map WHERE user = current_user() AND region = ...)</code></li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A healthcare analytics platform served 30 hospitals from one <code>patients</code> table. Each hospital's analysts must see only their own patients, and only clinicians may see names and dates of birth. A row filter checks a mapping table <code>hospital_access(user, hospital_id)</code>, and column masks on <code>name</code> and <code>dob</code> return values only for the <code>clinicians</code> group. Every tool (notebooks, SQL, dashboards, Genie) gets the same protection automatically, and onboarding a hospital means adding rows to the mapping table.
</div>

## Syntax

```sql
-- column mask
CREATE FUNCTION c.s.mask_email(email STRING) RETURNS STRING
  RETURN CASE WHEN is_account_group_member('pii_readers') THEN email
              ELSE concat('***@', split_part(email, '@', 2)) END;
ALTER TABLE c.s.customers ALTER COLUMN email SET MASK c.s.mask_email;
ALTER TABLE c.s.customers ALTER COLUMN email DROP MASK;

-- row filter
CREATE FUNCTION c.s.region_filter(region STRING) RETURNS BOOLEAN
  RETURN is_account_group_member('admins') OR region = 'IN';
ALTER TABLE c.s.customers SET ROW FILTER c.s.region_filter ON (region);
ALTER TABLE c.s.customers DROP ROW FILTER;
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a <code>pii_customers</code> table with personal data in the <code>governance</code> schema.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
6 customers from three regions.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.governance"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {DB}.pii_customers AS
    SELECT * FROM VALUES
      (1, 'Asha Rao',    'asha@example.com',  '+91 98450 11111', 'IN', 1200.0),
      (2, 'Ben Clark',   'ben@example.com',   '+44 7700 900123', 'UK',  800.0),
      (3, 'Chen Wei',    'chen@example.com',  '+971 50 123 4567', 'AE', 3100.0),
      (4, 'Divya Nair',  'divya@example.com', '+91 98450 22222', 'IN',  450.0),
      (5, 'Emma Stone',  'emma@example.com',  '+44 7700 900456', 'UK', 2200.0),
      (6, 'Farah Khan',  'farah@example.com', '+971 50 765 4321', 'AE',  950.0)
    AS t(customer_id, name, email, phone, region, lifetime_value)
""")
spark.table(f"{DB}.pii_customers").orderBy("customer_id").show(truncate=False)
me = spark.sql("SELECT current_user()").first()[0]
print("current user:", me)

## 1. Pseudonymize with hashing

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds an analytics-friendly copy where the email is replaced by a salted SHA-256 hash and the name and phone are dropped.<br><br>
<b>Why it matters</b><br>Analysts often need to count and join customers, not to know who they are. A stable hash keeps joins and distinct counts working. The salt (kept in a secret scope in real pipelines) makes it harder to reverse by hashing guessed emails.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Each customer has a 64-character <code>customer_key</code> instead of an email, and no name or phone.
</div>

In [0]:
from pyspark.sql import functions as F

SALT = "demo-salt-keep-real-salts-in-a-secret-scope"
pseudo = (spark.table(f"{DB}.pii_customers")
          .withColumn("customer_key", F.sha2(F.concat(F.lit(SALT), F.lower(F.trim("email"))), 256))
          .drop("name", "email", "phone"))
pseudo.orderBy("customer_id").show(truncate=40)

## 2. Dynamic view: mask columns and filter rows by user

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a view that masks the email and phone unless the user is a PII reader, and shows only rows for the regions a user may see. Here "PII readers" and the region access are checked with <code>current_user()</code>, so the cell also runs in plain Spark.<br><br>
<b>Why it matters</b><br>A dynamic view adapts its output to whoever runs the query. On Databricks, use <code>is_account_group_member('group')</code> instead of listing users. Users must be given access to the view, not the base table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
You aren't in the PII readers list, so email and phone are masked. You see only the <code>IN</code> region, because the view grants you <code>IN</code>.
</div>

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.pii_customers_dynamic AS
    SELECT customer_id,
           CASE WHEN current_user() IN ('privacy.officer@example.com') THEN name  ELSE concat(left(name, 1), '.') END AS name,
           CASE WHEN current_user() IN ('privacy.officer@example.com') THEN email ELSE concat('***@', substring_index(email, '@', -1)) END AS email,
           CASE WHEN current_user() IN ('privacy.officer@example.com') THEN phone ELSE concat(substring_index(phone, ' ', 1), ' *****') END AS phone,
           region, lifetime_value
    FROM {DB}.pii_customers
    WHERE region = CASE WHEN current_user() = '{me}' THEN 'IN' ELSE 'none' END
""")
spark.table(f"{DB}.pii_customers_dynamic").orderBy("customer_id").show(truncate=False)

## 3. Column mask on the table

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a masking function and attaches it to the <code>email</code> column of the table itself, then queries the table directly.<br><br>
<b>Why it matters</b><br>Unlike a view, a column mask protects the <b>table</b>: every query, from every tool, sees the masked value unless the function says otherwise. There's no way around it by querying the base table.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Column masks are a Unity Catalog feature that works on serverless compute. If they're not available in your workspace, the cell prints a message. Even as the table's owner, you see masked values, because you're not in the <code>pii_readers</code> group.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Emails appear as <code>***@example.com</code>. After dropping the mask, the real emails are back.
</div>

In [0]:
try:
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {DB}.pii_mask_email(email STRING)
        RETURNS STRING
        RETURN CASE WHEN is_account_group_member('pii_readers') THEN email
                    ELSE concat('***@', substring_index(email, '@', -1)) END
    """)
    spark.sql(f"ALTER TABLE {DB}.pii_customers ALTER COLUMN email SET MASK {DB}.pii_mask_email")
    print("with mask:")
    display(spark.table(f"{DB}.pii_customers").select("customer_id", "email"))
    spark.sql(f"ALTER TABLE {DB}.pii_customers ALTER COLUMN email DROP MASK")
    print("mask dropped, first email:", spark.table(f"{DB}.pii_customers").orderBy("customer_id").first()["email"])
except Exception as e:
    print("Column masks not available here:", type(e).__name__, str(e)[:200])

## 4. Row filter with a mapping table

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a mapping table <code>pii_region_access(user_email, region)</code> that gives you access to <code>UK</code>, a row filter function that checks it, and attaches the filter to the table.<br><br>
<b>Why it matters</b><br>Mapping tables keep access rules as <b>data</b>: granting a user a new region is an <code>INSERT</code>, not a code change. The row filter applies to every query on the table. The function parameter is named <code>row_region</code>, not <code>region</code>: inside the subquery, <code>region</code> would refer to the mapping table's column and match every row.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Row filters are a Unity Catalog feature that works on serverless compute. If unavailable, the cell prints a message.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
With the filter, only the two <code>UK</code> customers are visible. After adding <code>AE</code> to the mapping, four rows. After dropping the filter, all six.
</div>

In [0]:
try:
    spark.sql(f"CREATE OR REPLACE TABLE {DB}.pii_region_access (user_email STRING, region STRING)")
    spark.sql(f"INSERT INTO {DB}.pii_region_access VALUES ('{me}', 'UK')")
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {DB}.pii_region_filter(row_region STRING)
        RETURNS BOOLEAN
        RETURN is_account_group_member('admins_all_regions')
            OR EXISTS (SELECT 1 FROM {DB}.pii_region_access a WHERE a.user_email = current_user() AND a.region = row_region)
    """)
    spark.sql(f"ALTER TABLE {DB}.pii_customers SET ROW FILTER {DB}.pii_region_filter ON (region)")
    print("rows visible with filter:", spark.table(f"{DB}.pii_customers").count())
    spark.sql(f"INSERT INTO {DB}.pii_region_access VALUES ('{me}', 'AE')")
    print("after granting AE in the mapping table:", spark.table(f"{DB}.pii_customers").count())
    spark.sql(f"ALTER TABLE {DB}.pii_customers DROP ROW FILTER")
    print("filter dropped:", spark.table(f"{DB}.pii_customers").count())
except Exception as e:
    print("Row filters not available here:", type(e).__name__, str(e)[:200])

## 5. Which policies are on a table?

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Describes the table to show attached row filters and column masks, and lists them from <code>information_schema</code>.<br><br>
<b>Why it matters</b><br>When someone reports "missing rows" or "weird emails", the first check is whether a row filter or mask applies to them.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The table description (with row filter and mask information while they're attached). The <code>information_schema</code> views may be empty because both were dropped above.
</div>

In [0]:
display(spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.pii_customers"))
for view in ["row_filters", "column_masks"]:
    try:
        display(spark.sql(f"SELECT * FROM {CATALOG}.information_schema.{view} WHERE table_schema = 'governance'"))
    except Exception as e:
        print(f"information_schema.{view} not available here:", type(e).__name__)

## Under the hood

```
SELECT email FROM pii_customers   (user u)
  ├─ privileges checked (USE CATALOG, USE SCHEMA, SELECT)
  ├─ row filter:  WHERE pii_region_filter(region)         evaluated per row, as the querying user
  └─ column mask: pii_mask_email(email) AS email           evaluated per value, as the querying user
→ the same rewritten plan for notebooks, SQL warehouses, dashboards, Genie, JDBC/ODBC clients
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> filters and masks are expressions added to the query plan: keep the functions simple (avoid heavy subqueries) for performance.

In [0]:
spark.table(f"{DB}.pii_customers_dynamic").explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: dynamic view used, but users can still query the base table</b><br>
Revoke access to the table. Better: use row filters and column masks on the table itself.<br><br>
<b>⛔ Problem: masks that check individual users</b><br>Hard to maintain. Check groups with <code>is_account_group_member</code>, or use mapping tables.<br><br>
<b>⛔ Problem: unsalted hashes of emails</b><br>Easy to reverse by hashing a list of known emails. Use a secret salt (from a secret scope), and remember hashing isn't access control.<br><br>
<b>⛔ Problem: slow queries after adding a row filter</b><br>Complex functions run for every row. Keep filters simple, and index mapping tables by keeping them small.<br><br>
<b>⛔ Problem: "rows are missing" tickets</b><br>Check <code>DESCRIBE TABLE EXTENDED</code> for row filters and the user's group memberships first.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you implement column-level security in Unity Catalog?</b><br>
Create a SQL masking function that returns the real value for authorized users (usually checked with <code>is_account_group_member</code>) and a masked value for others, then attach it with <code>ALTER TABLE ... ALTER COLUMN ... SET MASK</code>. Alternatively, a dynamic view, but then access to the base table must be revoked.<br><br>

<b>🎤 2. How do you implement row-level security?</b><br>
With a row filter: a SQL function returning a boolean, attached with <code>ALTER TABLE ... SET ROW FILTER f ON (col)</code>. It often checks group membership or a mapping table of users and allowed values, so access changes are data changes.<br><br>

<b>🎤 3. Row filters and masks vs dynamic views: which do you prefer?</b><br>
Row filters and column masks, because they're enforced on the table for every query and tool, and there's no base table to forget to lock down. Dynamic views are still useful for computed or reshaped outputs, or where filters and masks aren't supported.<br><br>

<b>🎤 4. What is the difference between masking and pseudonymization?</b><br>
Masking hides values at query time based on who's asking, while the stored data is unchanged. Pseudonymization replaces values in the stored data (for example with a salted hash), so nobody can see the original from that dataset, but joins and counts still work.<br><br>

<b>🎤 5. Which functions are useful inside dynamic views and policies?</b><br>
<code>current_user()</code> to identify the user, and <code>is_account_group_member('group')</code> to check account-level group membership. Mapping tables can be referenced with <code>EXISTS</code> subqueries.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A team needs every query on <code>customers</code> to show emails only to members of the <code>pii_readers</code> group, regardless of the tool used. What should they implement?</b><br>
A. A temporary view<br>
B. A column mask on <code>email</code> using <code>is_account_group_member('pii_readers')</code><br>
C. A comment on the column<br>
D. A separate table per group<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which function returns whether the querying user belongs to an account-level group, for use in dynamic views and row filters?</b><br>
A. <code>current_group()</code><br>
B. <code>is_account_group_member()</code><br>
C. <code>has_privilege()</code><br>
D. <code>session_user_group()</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. How is a row filter attached to a table?</b><br>
A. <code>GRANT ROW FILTER ON TABLE t</code><br>
B. <code>ALTER TABLE t SET ROW FILTER f ON (col)</code><br>
C. <code>CREATE ROW POLICY ON t</code><br>
D. <code>ALTER TABLE t ADD CONSTRAINT f CHECK (col)</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a dynamic view that shows <code>lifetime_value</code> rounded to the nearest 1,000 for everyone except a list of finance users</li><li>(Databricks) Create a column mask for <code>phone</code> that shows only the country code (for example <code>+44 *****</code>) unless the user is in <code>pii_readers</code>, attach it, query, then drop it</li><li>(Databricks) Change the region mapping so that you see <code>IN</code> and <code>UK</code>, apply the row filter, and count the visible rows</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Coarsen a sensitive measure for non-finance users
spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.pii_customers_value AS
    SELECT customer_id, region,
           CASE WHEN current_user() IN ('cfo@example.com', 'finance.analyst@example.com') THEN lifetime_value
                ELSE ROUND(lifetime_value, -3) END AS lifetime_value
    FROM {DB}.pii_customers
""")
spark.table(f"{DB}.pii_customers_value").orderBy("customer_id").show()

In [0]:
# 2. Phone mask: keep only the country code
try:
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {DB}.pii_mask_phone(phone STRING) RETURNS STRING
        RETURN CASE WHEN is_account_group_member('pii_readers') THEN phone
                    ELSE concat(split_part(phone, ' ', 1), ' *****') END
    """)
    spark.sql(f"ALTER TABLE {DB}.pii_customers ALTER COLUMN phone SET MASK {DB}.pii_mask_phone")
    display(spark.table(f"{DB}.pii_customers").select("customer_id", "phone"))
    spark.sql(f"ALTER TABLE {DB}.pii_customers ALTER COLUMN phone DROP MASK")
except Exception as e:
    print("Column masks not available here:", type(e).__name__)

# 3. IN + UK through the mapping table
try:
    spark.sql(f"DELETE FROM {DB}.pii_region_access WHERE user_email = '{me}'")
    spark.sql(f"INSERT INTO {DB}.pii_region_access VALUES ('{me}', 'IN'), ('{me}', 'UK')")
    spark.sql(f"ALTER TABLE {DB}.pii_customers SET ROW FILTER {DB}.pii_region_filter ON (region)")
    print("visible rows (IN + UK):", spark.table(f"{DB}.pii_customers").count())
    spark.sql(f"ALTER TABLE {DB}.pii_customers DROP ROW FILTER")
except Exception as e:
    print("Row filters not available here:", type(e).__name__)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Pseudonymize (salted hash) when analysts don't need identities. Mask when some users do</li><li>Dynamic views: <code>current_user()</code>, <code>is_account_group_member()</code>, but lock the base table</li><li>Column masks and row filters protect the table itself, for every tool</li><li>Mapping tables make row-level access a data change, not a code change</li><li>Check <code>DESCRIBE TABLE EXTENDED</code> when rows or values look "wrong"</li></ul>
</div>

| Need | Use |
|---|---|
| Hide a column from some users | `ALTER COLUMN c SET MASK f` |
| Show only some rows | `SET ROW FILTER f ON (col)` |
| Per-user allowed values | Mapping table + `EXISTS` in the filter function |
| Group check | `is_account_group_member('g')` |
| Who is querying | `current_user()` |
| Irreversible de-identification | `sha2(concat(salt, value), 256)` |

## Git commit

```
git add 06_unity_catalog/07_pii_masking_row_filters.ipynb
git commit -m "add 06_07 pii masking and row filters notebook"
```